# Решения: Практика: признаки строки заказа

**Для преподавателя.** Секционный эталон урока и ДЗ; до сдачи ученикам не показывать.


Работаем как команда CRM маркетплейса: из трёх связанных таблиц нужно получить
объяснимые признаки, а не просто добиться вывода без ошибки. Перед каждой
операцией сформулируйте единицу наблюдения, ключ соединения и ожидаемое число
строк. После операции прочитайте assert как исполняемый контракт.

Сначала сделайте минимальный рабочий вариант, затем проверьте его на данных и
только после этого интерпретируйте результат. Не вводите метку churn: в этом
модуле мы строим и проверяем признаки, но не обучаем модель оттока.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

def find_csv(name):
    for path in (
        Path(name),
        Path("../") / name,
        Path("../../data") / name,
        Path("../data") / name,
        Path("../../../data") / name,
    ):
        if path.exists():
            return path.resolve()
    return "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/modules/08_05_shop_feature_engineering/data/" + name

orders = pd.read_csv(find_csv("orders_slim.csv"), parse_dates=["order_purchase_timestamp", "order_delivered_customer_date"])
customers = pd.read_csv(find_csv("customers_slim.csv"))
payments = pd.read_csv(find_csv("payments_slim.csv"))
assert len(orders) and len(customers) and len(payments)
assert orders["order_id"].is_unique and customers["customer_id"].is_unique
print(f"orders={len(orders)}, customers={len(customers)}, payments={len(payments)}")


## Урок. 1. Календарные признаки

In [ ]:
work=orders.copy()
work["order_month"]=work["order_purchase_timestamp"].dt.month
work["weekday"]=work["order_purchase_timestamp"].dt.weekday
assert work["weekday"].between(0,6).all()


## Урок. 2. Выходной день через lambda

In [ ]:
work["is_weekend"]=work["weekday"].apply(lambda d: int(d>=5))
assert set(work["is_weekend"])=={0,1}


## Урок. 3. Join с оплатой

In [ ]:
joined=work.merge(payments,on="order_id",how="left",validate="one_to_one")
assert len(joined)==len(work)


## Урок. 4. Срок доставки

In [ ]:
joined["days_to_deliver"]=(joined["order_delivered_customer_date"]-joined["order_purchase_timestamp"]).dt.days
assert joined["days_to_deliver"].dropna().ge(0).all()


## Урок. 5. Порог выброса

In [ ]:
p99=float(joined["days_to_deliver"].quantile(.99)); outlier_mask=joined["days_to_deliver"]>=p99
assert 1<=int(outlier_mask.sum())


## Урок. 6. Категория срока

In [ ]:
def delivery_band(days):
    if pd.isna(days): return "missing"
    if days<=7: return "fast"
    if days<=14: return "normal"
    return "slow"
joined["delivery_band"]=joined["days_to_deliver"].apply(delivery_band)
assert {"fast","normal","slow"} <= set(joined["delivery_band"]) <= {"missing","fast","normal","slow"}


## Урок. 7. Проверка формы и ключа

In [ ]:
quality_checks={"rows":len(joined)==len(orders),"unique_key":joined["order_id"].is_unique,"payment_complete":joined["payment_value"].notna().all(),"dates_nonnegative":joined["days_to_deliver"].dropna().ge(0).all()}
assert set(quality_checks.values())=={True}


## Урок. 8. Риск утечки времени

In [ ]:
LEAKAGE_NOTE="Момент расчёта определяет допустимость признака. Дата доставки и итоговый срок известны только после доставки, поэтому модель, принимающая решение при оформлении заказа, не должна видеть days_to_deliver. Для позднего описательного отчёта признак допустим; документация обязана назвать момент доступности."
assert len(LEAKAGE_NOTE)>=240


## ДЗ. 1. Средний чек по месяцам

In [ ]:
joined=orders.merge(payments,on="order_id"); joined["month"]=joined["order_purchase_timestamp"].dt.to_period("M").astype(str)
month_mean=joined.groupby("month")["payment_value"].mean()
assert len(month_mean)>=12


## ДЗ. 2. Доля card по weekday

In [ ]:
joined["weekday"]=joined["order_purchase_timestamp"].dt.weekday
joined["is_card"]=(joined["payment_type"]=="credit_card").astype(int)
weekday_card=joined.groupby("weekday")["is_card"].mean()
assert len(weekday_card)==7


## ДЗ. 3. Лог преобразований

In [ ]:
log_steps=["loaded three source tables","parsed purchase and delivery dates","created calendar features","joined payments one-to-one","checked output shape and key"]
assert len(log_steps)>=5


## ДЗ. 4. Challenge: функция признаков

In [ ]:
def add_order_features(frame):
    x=frame.copy(); x["order_month"]=x["order_purchase_timestamp"].dt.month; x["weekday"]=x["order_purchase_timestamp"].dt.weekday
    x["is_weekend"]=x["weekday"].ge(5).astype(int); x["days_to_deliver"]=(x["order_delivered_customer_date"]-x["order_purchase_timestamp"]).dt.days
    return x
featured=add_order_features(orders)
assert "order_month" not in orders


## ДЗ. 5. Challenge: решение по выбросам

In [ ]:
OUTLIER_NOTE="Порог p99 выделяет редкие сроки, но автоматически удалять их нельзя: это могут быть реальные бизнес-кейсы с высокой ценой для клиента. Сначала проверяем качество дат и статус заказа, затем сохраняем флаг выброса. Удаление допустимо только при доказанной технической ошибке и с записью в лог."
assert len(OUTLIER_NOTE)>=240
